# DQ-10 · inventory.csv
Checks: null rate, duplicates, FK → products, domain values, business rules (flags, rates), temporal.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── helpers ───────────────────────────────────────────────────────────────────
issues = []

def flag(label, mask_or_count, df=None, show_sample=True):
    count = int(mask_or_count.sum()) if hasattr(mask_or_count, 'sum') else int(mask_or_count)
    total = len(df) if df is not None else None
    pct   = f' ({count/total*100:.2f}%)' if total else ''
    status = '❌ ISSUE' if count > 0 else '✅ OK'
    print(f'{status}  {label}: {count:,}{pct}')
    if count > 0:
        issues.append(label)
        if show_sample and df is not None and hasattr(mask_or_count, 'sum'):
            sample = df[mask_or_count].head(3)
            print(sample.to_string(index=False))
    return count

def summary():
    print()
    if issues:
        print(f'══ {len(issues)} issue(s) found ══')
        for i in issues: print(f'  • {i}')
    else:
        print('══ All checks passed ══')


In [ ]:
inv   = pd.read_csv('inventory.csv', parse_dates=['snapshot_date'])
prods = pd.read_csv('products.csv')
print(f'Shape: {inv.shape}')
inv.head(3)

## 1. Null rate

In [ ]:
null_counts = inv.isnull().sum()
print(pd.DataFrame({'null_count': null_counts, 'null_%': (null_counts/len(inv)*100).round(2)}))

## 2. Duplicate (snapshot_date, product_id)

In [ ]:
flag('Duplicate (snapshot_date, product_id)', inv.duplicated(subset=['snapshot_date','product_id']), inv)

## 3. FK: product_id → products

In [ ]:
valid_prods = set(prods['product_id'])
flag('product_id not in products', ~inv['product_id'].isin(valid_prods), inv)

## 4. Domain values: flags

In [ ]:
for col in ['stockout_flag','overstock_flag','reorder_flag']:
    flag(f'{col} not in {{0,1}}', ~inv[col].isin([0,1]), inv)

# reorder_flag tất cả = 0?
print(f'\nreorder_flag unique values: {inv["reorder_flag"].unique()}')

## 5. Business rules: numeric columns

In [ ]:
flag('stock_on_hand < 0',   inv['stock_on_hand'] < 0,   inv)
flag('units_received < 0',  inv['units_received'] < 0,  inv)
flag('units_sold < 0',      inv['units_sold'] < 0,      inv)
flag('stockout_days < 0',   inv['stockout_days'] < 0,   inv)
flag('days_of_supply < 0',  inv['days_of_supply'] < 0,  inv)
flag('fill_rate < 0',       inv['fill_rate'] < 0,       inv)
flag('fill_rate > 1',       inv['fill_rate'] > 1,       inv)
flag('sell_through_rate < 0', inv['sell_through_rate'] < 0, inv)

## 6. Consistency: stockout_flag vs stockout_days

In [ ]:
# stockout_flag=1 nhưng stockout_days=0 (hoặc ngược lại)
flag('stockout_flag=1 but stockout_days=0',
     (inv['stockout_flag']==1) & (inv['stockout_days']==0), inv)
flag('stockout_flag=0 but stockout_days>0',
     (inv['stockout_flag']==0) & (inv['stockout_days']>0),  inv)

## 7. Temporal: snapshot_date là cuối tháng

In [ ]:
# Snapshot phải là ngày cuối tháng
import pandas as pd
inv['expected_eom'] = inv['snapshot_date'] + pd.offsets.MonthEnd(0)
flag('snapshot_date ≠ end of month', inv['snapshot_date'] != inv['expected_eom'], inv)
print(f'Date range: {inv["snapshot_date"].min().date()} → {inv["snapshot_date"].max().date()}')

## 8. Consistency: product_name, category, segment vs products

In [ ]:
prod_ref = prods[['product_id','product_name','category','segment']].rename(
    columns={'product_name':'pname_ref','category':'cat_ref','segment':'seg_ref'})
df = inv.merge(prod_ref, on='product_id', how='left')
flag('product_name mismatch', df['product_name'] != df['pname_ref'], df)
flag('category mismatch',     df['category']     != df['cat_ref'],   df)
flag('segment mismatch',      df['segment']      != df['seg_ref'],   df)

## Summary

In [ ]:
summary()